In [ ]:
!rm -rf aise26-18d4-distributed-training-sports
!git clone https://github.com/Andreachurchwell/aise26-18d4-distributed-training-sports.git
%cd aise26-18d4-distributed-training-sports
!ls


Cloning into 'aise26-18d4-distributed-training-sports'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 17 (delta 4), reused 15 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 6.48 KiB | 6.48 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/aise26-18d4-distributed-training-sports/aise26-18d4-distributed-training-sports/aise26-18d4-distributed-training-sports/aise26-18d4-distributed-training-sports
ddp_spawn.py	 README.md  requirements.txt  train.py
KNOWN_ISSUES.md  REPRO.md   SCALE_PLAN.md


In [ ]:
!pip -q install -r requirements.txt


In [ ]:
!ls


ddp_spawn.py	 README.md  requirements.txt  train.py
KNOWN_ISSUES.md  REPRO.md   SCALE_PLAN.md


In [ ]:
import os
import numpy as np
import pandas as pd

os.makedirs("data", exist_ok=True)

rng = np.random.default_rng(42)
n_rows = 3000

home_rest_days = rng.integers(0, 4, size=n_rows)
away_rest_days = rng.integers(0, 4, size=n_rows)
home_elo = rng.normal(1500, 80, size=n_rows)
away_elo = rng.normal(1500, 80, size=n_rows)

logit = (home_elo - away_elo) / 200 + (home_rest_days - away_rest_days) * 0.15
p_home_win = 1 / (1 + np.exp(-logit))
win = (rng.random(n_rows) < p_home_win).astype(int)

df = pd.DataFrame({
    "home_rest_days": home_rest_days,
    "away_rest_days": away_rest_days,
    "home_elo": home_elo,
    "away_elo": away_elo,
    "win": win,
})

df.to_csv("data/nba_team_games.csv", index=False)
print("Wrote numeric-only CSV with columns:", list(df.columns))
print(df.head())

Wrote numeric-only CSV with columns: ['home_rest_days', 'away_rest_days', 'home_elo', 'away_elo', 'win']
   home_rest_days  away_rest_days     home_elo     away_elo  win
0               0               3  1584.510847  1498.392436    1
1               3               2  1439.582446  1373.262693    0
2               2               3  1280.837350  1407.095184    1
3               1               3  1358.643048  1410.274715    0
4               1               2  1613.730158  1561.128658    1


In [ ]:
!python train.py --cpu


step=5 loss=5.7925 eff_batch=8
step=10 loss=0.0853 eff_batch=8
step=15 loss=1.0062 eff_batch=8
step=20 loss=0.0011 eff_batch=8
step=25 loss=0.0090 eff_batch=8
step=30 loss=0.2888 eff_batch=8
step=35 loss=0.0872 eff_batch=8
step=40 loss=0.4923 eff_batch=8
step=45 loss=2.8153 eff_batch=8
step=50 loss=2.7285 eff_batch=8
Done.
metrics written to: metrics.csv


In [ ]:
!torchrun --standalone --nproc_per_node=2 train.py --cpu


W0130 22:49:03.681000 6603 torch/distributed/run.py:803] 
W0130 22:49:03.681000 6603 torch/distributed/run.py:803] *****************************************
W0130 22:49:03.681000 6603 torch/distributed/run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0130 22:49:03.681000 6603 torch/distributed/run.py:803] *****************************************
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1[Gloo] Rank 1 is connected to 
1 peer ranks. Expected number of connected peer ranks is : 1
step=5 loss=6.1100 eff_batch=16
step=10 loss=5.7858 eff_batch=16
step=15 loss=0.9769 eff_batch=16
step=20 loss=0.0433 eff_batch=16
step=25 loss=4.1932 eff_batch=16
step=30 loss=1.8188 eff_batch=16
step=35 loss=0.8679 eff_batch=16
step=40 loss=1.3134 eff_batch=16
step=45 loss=0.3430 eff_batch=16

In [ ]:
!tail -n 8 metrics.csv


43,1.1043779850006104,16,2,4,1769813350
44,0.3520079255104065,16,2,4,1769813350
45,0.34303218126296997,16,2,4,1769813350
46,0.48283737897872925,16,2,4,1769813350
47,1.2753407955169678,16,2,4,1769813350
48,0.6081498861312866,16,2,4,1769813350
49,1.0477144718170166,16,2,4,1769813350
50,1.6138286590576172,16,2,4,1769813350


In [ ]:
!ls -l metrics.csv


-rw-r--r-- 1 root root 2097 Jan 30 22:49 metrics.csv


In [ ]:
!ls -l metrics.csv
!head -n 1 metrics.csv
!tail -n 3 metrics.csv


-rw-r--r-- 1 root root 2097 Jan 30 22:49 metrics.csv
step,loss,effective_batch_size,world_size,accum_steps,timestamp
48,0.6081498861312866,16,2,4,1769813350
49,1.0477144718170166,16,2,4,1769813350
50,1.6138286590576172,16,2,4,1769813350


In [ ]:
from google.colab import files
files.download("metrics.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("data/nba_team_games.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>